# Sistema de Recomendação em Produção — Sam's Club (Walmart)

Este notebook documenta o **caminho de produção (serving)** do sistema de recomendação.

O modelo em produção é um **NeuMF** (*Neural Collaborative Filtering*, GMF + MLP — He
et al., 2017) treinado com objetivo de **ranking** e combinado com um **prior de
popularidade** num recomendador **híbrido popularity-first**:

```
score = z(log1p(popularidade)) + alpha * z(NeuMF)
```

O treino e o registro do modelo pertencem ao pipeline MLOps
(`src/scripts/otimizacao_modelo.py`, orquestrado pelo DVC). Aqui **consumimos os
artefatos versionados** em `models/` e demonstramos o que um sistema em produção
precisa garantir:

1. Deploy determinístico a partir dos artefatos salvos (modelo + encoders + config);
2. Avaliação de **ranking** no protocolo *leave-last-out* contra baselines honestos;
3. Geração do **top-7** personalizado, excluindo o que o cliente já comprou;
4. Tratamento de ***cold start*** (cliente fora da base de treino).

## 1. Configuração do ambiente

In [1]:
import sys
from pathlib import Path

# Localiza a raiz do projeto (onde está o pyproject.toml) para importar `src.*`.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import joblib
import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

from src.config.paths import DADOS_TRATADOS, MODELS_DIR, REPORTS_DIR
from src.config.model_utils import (
    TOP_K,
    RankingCheckpoint,
    build_item_catalog,
    build_training_pairs,
    categories_for_products,
    combine_scores,
    compute_popularity_prior,
    create_ranking_model,
    evaluate_hybrid,
    evaluate_most_popular,
    evaluate_random_baseline,
    evaluate_ranking,
    recommend_products,
    score_catalog,
    select_alpha,
    to_model_inputs,
    train_ranking_model,
    train_validation_test_split,
)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print("Projeto:", ROOT)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

Projeto: C:\Users\flavi\Documents\GitHub\Sistema_Recomendacao_Arquitetura-MLOps
TensorFlow: 2.19.0 | Keras: 3.9.0


## 2. Carregamento dos artefatos de produção

Um deploy precisa ser **reprodutível**: carregamos exatamente os artefatos produzidos
pelo pipeline — o modelo `.keras`, os quatro `LabelEncoder` e a configuração do
híbrido (`alpha` + vetor de popularidade). Nada é retreinado aqui.

In [2]:
dados = pd.read_parquet(DADOS_TRATADOS)
encoders = {
    nome: joblib.load(MODELS_DIR / f"{nome}_encoder_otimizado.pkl")
    for nome in ["customer", "product", "category", "subcategory"]
}
modelo = tf.keras.models.load_model(MODELS_DIR / "best_model_recomendacao_otimizado.keras")
config_hibrido = joblib.load(MODELS_DIR / "hybrid_config_otimizado.pkl")

alpha = float(config_hibrido["alpha"])
prior_popularidade = config_hibrido["popularity"]
num_produtos = len(encoders["product"].classes_)
catalogo = build_item_catalog(
    dados, encoders["product"], encoders["category"], encoders["subcategory"]
)

train_pairs, val_pairs, test_pairs = train_validation_test_split(
    dados, encoders["customer"], encoders["product"]
)

print("Interações:", dados.shape)
print(f"Clientes: {len(encoders['customer'].classes_)} | Produtos: {num_produtos} "
      f"| Categorias: {len(encoders['category'].classes_)}")
print(f"Pares -> treino: {len(train_pairs)} | validação: {len(val_pairs)} | teste: {len(test_pairs)}")
print(f"alpha do híbrido (produção): {alpha}")

Interações: (9800, 18)
Clientes: 793 | Produtos: 1861 | Categorias: 3
Pares -> treino: 8171 | validação: 776 | teste: 776
alpha do híbrido (produção): 0.0


## 3. Avaliação de ranking (*leave-last-out*)

Protocolo padrão em filtragem colaborativa implícita: o **último** item de cada
cliente é o teste, o **penúltimo** é a validação e o restante é o treino. Para cada
cliente ranqueamos o **catálogo completo**, **excluímos os itens já vistos** e
comparamos o top-k com o item reservado (*held-out*).

Comparamos quatro políticas:

| Política | O que é |
|---|---|
| **Híbrido (produção)** | `z(log1p(pop)) + alpha * z(NeuMF)` com o `alpha` escolhido na validação |
| **NeuMF (neural puro)** | apenas o sinal personalizado da rede |
| **Mais populares** | baseline não personalizado (ranking determinístico) |
| **Aleatório** | esperança analítica do sorteio uniforme |

In [3]:
metricas = {
    "Híbrido (produção)": evaluate_hybrid(
        modelo, test_pairs, train_pairs, catalogo, prior_popularidade, alpha, k=TOP_K
    ),
    "NeuMF (neural puro)": evaluate_ranking(modelo, test_pairs, train_pairs, catalogo, k=TOP_K),
    "Mais populares": evaluate_most_popular(test_pairs, train_pairs, num_produtos, k=TOP_K),
    "Aleatório (esperança)": evaluate_random_baseline(test_pairs, train_pairs, num_produtos, k=TOP_K),
}

linhas = []
for nome, m in metricas.items():
    linhas.append({
        "Política": nome,
        f"NDCG@{TOP_K}": m.get(f"ndcg@{TOP_K}", np.nan),
        f"MAP@{TOP_K}": m.get(f"map@{TOP_K}", np.nan),
        f"HitRate@{TOP_K}": m.get(f"hit_rate@{TOP_K}", np.nan),
        f"Precision@{TOP_K}": m.get(f"precision@{TOP_K}", np.nan),
        f"Recall@{TOP_K}": m.get(f"recall@{TOP_K}", np.nan),
        f"F1@{TOP_K}": m.get(f"f1@{TOP_K}", np.nan),
    })

tabela = pd.DataFrame(linhas).set_index("Política").round(4).fillna("—")
tabela

,NDCG@7,MAP@7,HitRate@7,Precision@7,Recall@7,F1@7
Política,,,,,,
Híbrido (produção),0.0027,0.002,0.0052,0.0007,0.0052,0.0013
NeuMF (neural puro),0.0009,0.0004,0.0026,0.0004,0.0026,0.0006
Mais populares,0.0027,0.002,0.0052,0.0007,0.0052,0.0013
Aleatório (esperança),—,—,0.0038,0.0005,0.0038,—


### Leitura honesta dos resultados

Nesta base (~12 itens distintos por cliente em um catálogo de ~1.900), o baseline
**mais-populares** é muito forte. O `alpha` é escolhido na validação por uma política
**conservadora** (só adota personalização se ela superar o prior por margem mínima);
quando isso não acontece, o híbrido **degenera explicitamente em `alpha=0`** e
reproduz o baseline popular — em vez de forçar um ganho que não existe. É essa
honestidade que torna as métricas auditáveis.

Os traços (**—**) marcam métricas sem forma fechada para o baseline **aleatório**:
reportamos apenas a esperança **analítica exata** de Hit-Rate, Precision e Recall, em
vez de aproximar NDCG/MAP/F1 por simulação.

In [4]:
base = metricas["Mais populares"][f"ndcg@{TOP_K}"]
ganho = metricas["Híbrido (produção)"][f"ndcg@{TOP_K}"] / base - 1
print(f"NDCG@{TOP_K} do híbrido sobre o baseline 'mais populares': {ganho:+.1%}")
print(f"NDCG@{TOP_K} do NeuMF puro: {metricas['NeuMF (neural puro)'][f'ndcg@{TOP_K}']:.4f}")
print(f"Hit-Rate@{TOP_K} do aleatório (referência baixa): "
      f"{metricas['Aleatório (esperança)'][f'hit_rate@{TOP_K}']:.4f}")
print()
print("Observação: com alpha=0 o híbrido é, por construção, idêntico ao baseline popular.")

NDCG@7 do híbrido sobre o baseline 'mais populares': +0.0%
NDCG@7 do NeuMF puro: 0.0009
Hit-Rate@7 do aleatório (referência baixa): 0.0038

Observação: com alpha=0 o híbrido é, por construção, idêntico ao baseline popular.


## 4. Serviço de recomendação (top-7)

A função `recommend_products` encapsula o serving: pontua o catálogo, aplica o score
híbrido, remove o que o cliente já comprou e devolve o top-k com metadados.

In [5]:
CLIENTE = "Irene Maddox"

historico = (
    dados[dados["Customer Name"] == CLIENTE]
    .sort_values("Order Date")
    .tail(5)[["Order Date", "Product ID", "Product Name", "Category", "Sub-Category"]]
)
print(f"Últimas compras de {CLIENTE}:")
display(historico)

recomendacoes = recommend_products(
    modelo, CLIENTE, dados, encoders["customer"], encoders["product"], catalogo,
    top_k=TOP_K, popularity=prior_popularidade, alpha=alpha,
)
recomendacoes

Últimas compras de Irene Maddox:


,Order Date,Product ID,Product Name,Category,Sub-Category
2087,23/06/2017,OFF-PA-10003129,Tops White Computer Printout Paper,Office Supplies,Paper
2086,23/06/2017,OFF-ST-10000060,Fellowes Bankers Box Staxonsteel Drawer File/S...,Office Supplies,Storage
2085,23/06/2017,OFF-LA-10000248,Avery 52,Office Supplies,Labels
2084,23/06/2017,OFF-LA-10003510,Avery 4027 File Folder Labels for Dot Matrix P...,Office Supplies,Labels
2088,23/06/2017,OFF-FA-10004076,Translucent Push Pins by OIC,Office Supplies,Fasteners


,ranking,Product ID,Product Name,Category,Sub-Category,score
0,1,OFF-PA-10001970,Xerox 1881,Office Supplies,Paper,2.667958
1,2,TEC-AC-10003832,Logitech P710e Mobile Speakerphone,Technology,Accessories,2.667958
2,3,OFF-BI-10004140,Avery Non-Stick Binders,Office Supplies,Binders,2.253192
3,4,TEC-AC-10003628,Logitech 910-002974 M325 Wireless Mouse for We...,Technology,Accessories,2.253192
4,5,OFF-BI-10000977,Ibico Plastic Spiral Binding Combs,Office Supplies,Binders,2.253192
5,6,FUR-CH-10002647,"Situations Contoured Folding Chairs, 4/Set",Furniture,Chairs,2.094879
6,7,TEC-AC-10002049,Logitech G19 Programmable Gaming Keyboard,Technology,Accessories,2.094879


## 5. *Cold start* — cliente fora da base de treino

Um cliente novo não tem histórico nem embedding. O sistema precisa degradar com
elegância: cai no baseline **mais populares** (nunca em erro).

In [6]:
cliente_novo = "Cliente Novo (sem histórico)"
try:
    encoders["customer"].transform([cliente_novo])
    print("Cliente encontrado na base.")
except ValueError:
    print(f"'{cliente_novo}' não está na base -> acionando o fallback de cold start.")

recomendacoes_novo = recommend_products(
    modelo, cliente_novo, dados, encoders["customer"], encoders["product"], catalogo,
    top_k=TOP_K, popularity=prior_popularidade, alpha=alpha,
)
recomendacoes_novo

'Cliente Novo (sem histórico)' não está na base -> acionando o fallback de cold start.


,ranking,Product ID,Product Name,Category,Sub-Category,score
0,1,OFF-PA-10001970,Xerox 1881,Office Supplies,Paper,19.0
1,2,TEC-AC-10003832,Logitech P710e Mobile Speakerphone,Technology,Accessories,18.0
2,3,FUR-FU-10004270,"Eldon Image Series Desk Accessories, Burgundy",Furniture,Furnishings,16.0
3,4,TEC-AC-10003628,Logitech 910-002974 M325 Wireless Mouse for We...,Technology,Accessories,15.0
4,5,FUR-CH-10002647,"Situations Contoured Folding Chairs, 4/Set",Furniture,Chairs,15.0
5,6,TEC-AC-10002049,Logitech G19 Programmable Gaming Keyboard,Technology,Accessories,15.0
6,7,FUR-CH-10003774,"Global Wood Trimmed Manager's Task Chair, Khaki",Furniture,Chairs,14.0


## 6. Conclusão

- O sistema é **reprodutível**: o deploy consome apenas artefatos versionados (DVC),
  sem re-treino.
- A avaliação usa o protocolo **leave-last-out** com **baselines honestos**; quando a
  personalização não é vantajosa, o híbrido assume o prior de popularidade (`alpha=0`).
- O *cold start* é tratado com *fallback* para os itens mais populares.

Próximos passos operacionais: expor `recommend_products` como serviço (API/worker),
monitorar *drift* do catálogo e reavaliar o `alpha` periodicamente na validação.